# Query Yahoo Fantasy Iceberg Tables

This notebook demonstrates how to load and query the Yahoo Fantasy data stored in Iceberg tables.

The notebook correctly handles Windows path resolution by:
1. Finding the project root from `pyproject.toml`
2. Using absolute paths for the Iceberg catalog
3. Using the correct warehouse location

In [28]:
from pathlib import Path
import sys

import polars as pl
from pyiceberg.catalog import load_catalog

In [29]:
# Configure Polars display
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [37]:
# Find project root from cwd by walking up to pyproject.toml
cwd = Path.cwd()

def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            if current.name.lower() == "examples":
                raise RuntimeError("Invalid project root resolved to 'examples'.")
            return current
        current = current.parent
    raise RuntimeError("Cannot locate project root. Expected pyproject.toml in repository root.")

project_root = find_project_root(cwd)

# Ensure all relative table paths resolve from repo root.
import os
os.chdir(project_root)

# Use absolute, URI-safe paths (important on Windows).
CATALOG_DB_PATH = (project_root / "iceberg_catalog.db").resolve()
WAREHOUSE_PATH = (project_root / "warehouse").resolve()
CATALOG_URI = f"sqlite:///{CATALOG_DB_PATH.as_posix()}"
WAREHOUSE_URI = WAREHOUSE_PATH.as_uri()

# Force fresh catalog load: clear cached modules
import sys
for mod_name in list(sys.modules.keys()):
    if 'pyiceberg' in mod_name:
        del sys.modules[mod_name]

# Now reload catalog with fresh connection
from pyiceberg.catalog import load_catalog as fresh_load_catalog
catalog = fresh_load_catalog(
    "yahoo",
    type="sql",
    uri=CATALOG_URI,
    warehouse=WAREHOUSE_URI,
    )

print("✓ Catalog loaded.")
print("  CWD:", Path.cwd())
print("  Catalog URI:", CATALOG_URI)
print("  Warehouse path:", WAREHOUSE_PATH)
print("  Warehouse URI:", WAREHOUSE_URI)

✓ Catalog loaded.
  CWD: C:\Users\EricTruett\nfl
  Catalog URI: sqlite:///C:/Users/EricTruett/nfl/iceberg_catalog.db
  Warehouse path: C:\Users\EricTruett\nfl\warehouse
  Warehouse URI: file:///C:/Users/EricTruett/nfl/warehouse


In [31]:
# Clear any cached catalog state and verify database exists
import os
if not (project_root / "iceberg_catalog.db").exists():
    print("⚠ Catalog database not found at:", project_root / "iceberg_catalog.db")
    print("  Creating fresh catalog...")
    # The catalog will auto-create the database on first connection
else:
    print("✓ Catalog database exists, using it")

✓ Catalog database exists, using it


In [38]:
# List available tables (with Windows-safe catalog metadata normalization)
from pathlib import Path
import re
import sqlite3

def _to_repo_relative_path(location: str, project_root: Path) -> str:
    value = location.replace('\\', '/')

    # file:///C:/... -> C:/...
    if value.lower().startswith("file:///"):
        value = value[8:]

    # /C:/... -> C:/...
    if re.match(r"^/[A-Za-z]:/", value):
        value = value[1:]

    root_prefix = project_root.as_posix().rstrip("/") + "/"
    if value.startswith(root_prefix):
        return value[len(root_prefix):]

    return value

def normalize_catalog_metadata_locations(catalog_db_path: Path, project_root: Path) -> int:
    conn = sqlite3.connect(str(catalog_db_path))
    try:
        cur = conn.cursor()
        rows = cur.execute(
            "SELECT catalog_name, table_namespace, table_name, metadata_location, previous_metadata_location FROM iceberg_tables"
        ).fetchall()

        updates: list[tuple[str, str | None, str, str, str]] = []
        for catalog_name, table_namespace, table_name, metadata_location, previous_metadata_location in rows:
            new_meta = (
                _to_repo_relative_path(metadata_location, project_root)
                if metadata_location
                else metadata_location
            )
            new_prev = (
                _to_repo_relative_path(previous_metadata_location, project_root)
                if previous_metadata_location
                else previous_metadata_location
            )
            if new_meta != metadata_location or new_prev != previous_metadata_location:
                updates.append((new_meta, new_prev, catalog_name, table_namespace, table_name))

        if updates:
            cur.executemany(
                """
                UPDATE iceberg_tables
                SET metadata_location = ?, previous_metadata_location = ?
                WHERE catalog_name = ? AND table_namespace = ? AND table_name = ?
                """,
                updates,
            )
            conn.commit()

        return len(updates)
    finally:
        conn.close()

def list_tables(namespace: str) -> list[str]:
    try:
        return [f"{ns}.{tbl}" for ns, tbl in catalog.list_tables(namespace)]
    except Exception as exc:
        print(f"Could not list tables for namespace '{namespace}': {exc}")
        return []

def _latest_metadata_file(table_dir: Path) -> Path | None:
    metadata_dir = table_dir / "metadata"
    if not metadata_dir.exists():
        return None
    metadata_files = sorted(
        metadata_dir.glob("*.metadata.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return metadata_files[0] if metadata_files else None

def register_tables_from_warehouse(warehouse: Path, namespaces: list[str]) -> int:
    registered = 0
    for namespace in namespaces:
        ns_dir = warehouse / namespace
        if not ns_dir.exists():
            continue

        try:
            catalog.create_namespace(namespace)
        except Exception:
            pass

        for table_dir in [p for p in ns_dir.iterdir() if p.is_dir()]:
            table_identifier = f"{namespace}.{table_dir.name}"
            try:
                catalog.load_table(table_identifier)
                continue
            except Exception:
                pass

            metadata_file = _latest_metadata_file(table_dir)
            if metadata_file is None:
                continue

            # Use repo-relative metadata path for best Windows compatibility.
            metadata_rel = metadata_file.resolve().relative_to(project_root).as_posix()
            try:
                catalog.register_table(table_identifier, metadata_rel)
                registered += 1
                print(f"Registered {table_identifier}")
            except Exception as exc:
                print(f"Could not register {table_identifier}: {exc}")

    return registered

fixed_count = normalize_catalog_metadata_locations(CATALOG_DB_PATH, project_root)
if fixed_count > 0:
    print(f"Normalized metadata locations for {fixed_count} catalog row(s).")
    from pyiceberg.catalog import load_catalog as fresh_load_catalog
    catalog = fresh_load_catalog(
        "yahoo",
        type="sql",
        uri=CATALOG_URI,
        warehouse=WAREHOUSE_URI,
    )

namespaces = ["yahoo_common", "yhnfl"]
print("Available tables:\n")

all_tables: dict[str, list[str]] = {}
for ns in namespaces:
    all_tables[ns] = list_tables(ns)

if not any(all_tables.values()):
    print("\nNo tables found in catalog. Attempting metadata registration from warehouse...")
    registered_count = register_tables_from_warehouse(WAREHOUSE_PATH, namespaces)
    if registered_count > 0:
        print(f"\nRegistered {registered_count} table(s). Reloading table list...\n")
        for ns in namespaces:
            all_tables[ns] = list_tables(ns)
    else:
        print("\nNo registerable metadata files were found.")

for ns in namespaces:
    print(f"{ns}:")
    if all_tables[ns]:
        for t in all_tables[ns]:
            print(f"  - {t}")
    else:
        print("  (no tables found)")

Normalized metadata locations for 11 catalog row(s).
Available tables:

yahoo_common:
  - yahoo_common.draft_pick
  - yahoo_common.league
  - yahoo_common.player
  - yahoo_common.scoring_rule
  - yahoo_common.stat_category
  - yahoo_common.team
  - yahoo_common.transaction
yhnfl:
  - yhnfl.matchups
  - yhnfl.player_stats_weekly
  - yhnfl.roster_entries
  - yhnfl.standings


In [33]:
# Utility functions for loading tables
def load_table_as_polars(table_identifier: str) -> pl.DataFrame:
    """Load an Iceberg table into a Polars DataFrame."""
    table = catalog.load_table(table_identifier)
    arrow_table = table.scan().to_arrow()
    return pl.from_arrow(arrow_table)

def maybe_load(table_identifier: str) -> pl.DataFrame | None:
    """Try to load a table, return None if it fails."""
    try:
        return load_table_as_polars(table_identifier)
    except Exception as exc:
        print(f"Skipping '{table_identifier}': {exc}")
        return None

## Example 1: League and Team Information

Load league and team data to see league context per team.

In [39]:
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

if league_df is not None and team_df is not None:
    league_team = (
        team_df.join(
            league_df.select(["league_key", "season", "league_name"]),
            on="league_key",
            how="left",
        )
        .select(["league_key", "season", "league_name", "team_key", "team_name", "owner_name"])
        .sort(["season", "league_key", "team_key"])
    )
    print("League and Team Information:")
    print(league_team)
else:
    print("Required tables are not available.")

League and Team Information:
shape: (12, 6)
┌──────────────┬────────┬─────────────┬───────────────────┬───────────────────────┬─────────────┐
│ league_key   ┆ season ┆ league_name ┆ team_key          ┆ team_name             ┆ owner_name  │
│ ---          ┆ ---    ┆ ---         ┆ ---               ┆ ---                   ┆ ---         │
│ str          ┆ i64    ┆ str         ┆ str               ┆ str                   ┆ str         │
╞══════════════╪════════╪═════════════╪═══════════════════╪═══════════════════════╪═════════════╡
│ 461.l.717896 ┆ 2025   ┆ Dilly Dilly ┆ 461.l.717896.t.1  ┆ DaSaints              ┆ Reggy       │
│ 461.l.717896 ┆ 2025   ┆ Dilly Dilly ┆ 461.l.717896.t.10 ┆ Kyler the Creator     ┆ Alexander   │
│ 461.l.717896 ┆ 2025   ┆ Dilly Dilly ┆ 461.l.717896.t.11 ┆ Action is the Juice   ┆ Eric        │
│ 461.l.717896 ┆ 2025   ┆ Dilly Dilly ┆ 461.l.717896.t.12 ┆ oooutzs               ┆ Arlo        │
│ 461.l.717896 ┆ 2025   ┆ Dilly Dilly ┆ 461.l.717896.t.2  ┆ Alameda All-St

## Example 2: NFL League Standings

Load standings data to rank teams by wins and points for.

In [40]:
standings_df = maybe_load("yhnfl.standings")

if standings_df is not None:
    standings_summary = (
        standings_df
        .sort(["season", "wins", "points_for"], descending=[False, True, True])
        .select([
            "league_key",
            "season",
            "team_key",
            "rank",
            "wins",
            "losses",
            "ties",
            "points_for",
            "points_against",
        ])
    )
    print("NFL League Standings:")
    print(standings_summary)
else:
    print("Table yhnfl.standings not found.")

NFL League Standings:
shape: (12, 9)
┌──────────────┬────────┬─────────────────┬──────┬───┬────────┬──────┬────────────┬────────────────┐
│ league_key   ┆ season ┆ team_key        ┆ rank ┆ … ┆ losses ┆ ties ┆ points_for ┆ points_against │
│ ---          ┆ ---    ┆ ---             ┆ ---  ┆   ┆ ---    ┆ ---  ┆ ---        ┆ ---            │
│ str          ┆ i64    ┆ str             ┆ i64  ┆   ┆ i64    ┆ i64  ┆ f64        ┆ f64            │
╞══════════════╪════════╪═════════════════╪══════╪═══╪════════╪══════╪════════════╪════════════════╡
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 4    ┆ … ┆ 3      ┆ 0    ┆ 1854.52    ┆ 1540.48        │
│              ┆        ┆ 6               ┆      ┆   ┆        ┆      ┆            ┆                │
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 1    ┆ … ┆ 4      ┆ 0    ┆ 1838.4     ┆ 1672.76        │
│              ┆        ┆ 11              ┆      ┆   ┆        ┆      ┆            ┆                │
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 3    ┆ … ┆

## Example 3: Weekly Fantasy Points by Team

This combines `yhnfl.player_stats_weekly` with `yhnfl.roster_entries` to summarize total fantasy points by week and team.

In [47]:
stats_df = maybe_load("yhnfl.player_stats_weekly")
roster_df = maybe_load("yhnfl.roster_entries")
matchups_df = maybe_load("yhnfl.matchups")

if stats_df is not None and roster_df is not None:
    points_available = (
        stats_df.select(pl.max("fantasy_points").alias("max_fp")).item() is not None
        and stats_df.select(pl.max("fantasy_points").alias("max_fp")).item() > 0
    ) or (
        roster_df.select(pl.max("points").alias("max_points")).item() is not None
        and roster_df.select(pl.max("points").alias("max_points")).item() > 0
    )

    if points_available:
        weekly_points = (
            stats_df.join(
                roster_df.select(["league_key", "week", "team_key", "player_key"]),
                on=["league_key", "week", "player_key"],
                how="left",
            )
            .group_by(["league_key", "season", "week", "team_key"])
            .agg([
                pl.sum("fantasy_points").alias("team_points"),
                pl.n_unique("player_key").alias("players_counted"),
            ])
            .sort(["season", "week", "team_points"], descending=[False, False, True])
        )
        print("Weekly Fantasy Points by Team (from player/roster stats):")
        print(weekly_points)
    elif matchups_df is not None:
        weekly_points = (
            matchups_df
            .select(["league_key", "season", "week", "team_key", pl.col("points").alias("team_points")])
            .sort(["season", "week", "team_points"], descending=[False, False, True])
        )
        print("Weekly Fantasy Points by Team (fallback from matchups.points):")
        print(weekly_points)
    else:
        print("Weekly point values are unavailable in player/roster tables, and matchups table is missing.")
else:
    print("Required NFL weekly tables not found.")

Weekly Fantasy Points by Team (fallback from matchups.points):
shape: (196, 5)
┌──────────────┬────────┬──────┬───────────────────┬─────────────┐
│ league_key   ┆ season ┆ week ┆ team_key          ┆ team_points │
│ ---          ┆ ---    ┆ ---  ┆ ---               ┆ ---         │
│ str          ┆ i64    ┆ i64  ┆ str               ┆ f64         │
╞══════════════╪════════╪══════╪═══════════════════╪═════════════╡
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.1  ┆ 154.76      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.5  ┆ 132.18      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.6  ┆ 123.84      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.4  ┆ 122.72      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.8  ┆ 119.58      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.2  ┆ 118.36      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.7  ┆ 111.4       │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.9  ┆ 105.38      │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ 461.l.717896.t.10

## Example 4: Player-Level Aggregation by Position

This view builds player-week records from `yhnfl.roster_entries` + `yhnfl.player_stats_weekly`, then summarizes points by lineup position.

In [49]:
print('stats_df schema:', stats_df.schema if stats_df is not None else None)
print('roster_df schema:', roster_df.schema if roster_df is not None else None)
if stats_df is not None:
    print('\nfantasy_points summary:')
    print(stats_df.select([
        pl.len().alias('rows'),
        pl.sum('fantasy_points').alias('sum_fantasy_points'),
        pl.mean('fantasy_points').alias('avg_fantasy_points'),
        pl.min('fantasy_points').alias('min_fantasy_points'),
        pl.max('fantasy_points').alias('max_fantasy_points'),
    ]))

if roster_df is not None:
    print('\nroster points summary:')
    print(roster_df.select([
        pl.len().alias('rows'),
        pl.sum('points').alias('sum_points'),
        pl.mean('points').alias('avg_points'),
        pl.min('points').alias('min_points'),
        pl.max('points').alias('max_points'),
    ]))

if stats_df is not None:
    print('\nSample stats rows:')
    print(stats_df.select(['league_key', 'week', 'player_key', 'fantasy_points', 'stats']).head(5))

stats_df schema: Schema({'league_key': String, 'season': Int64, 'week': Int64, 'player_key': String, 'fantasy_points': Float64, 'status': String, 'bye_week': Int64, 'stats': List(String)})
roster_df schema: Schema({'league_key': String, 'season': Int64, 'week': Int64, 'team_key': String, 'player_key': String, 'selected_position': String, 'is_starting': Boolean, 'points': Float64})

fantasy_points summary:
shape: (1, 5)
┌──────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ rows ┆ sum_fantasy_points ┆ avg_fantasy_points ┆ min_fantasy_points ┆ max_fantasy_points │
│ ---  ┆ ---                ┆ ---                ┆ ---                ┆ ---                │
│ u32  ┆ f64                ┆ f64                ┆ f64                ┆ f64                │
╞══════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ 2968 ┆ 0.0                ┆ 0.0                ┆ 0.0                ┆ 0.0                │
└──────┴───────────

In [50]:
player_weekly_points = None
position_weekly_points = None
team_position_weekly_points = None

if roster_df is not None and stats_df is not None:
    player_weekly_points = (
        roster_df
        .select([
            "league_key",
            "season",
            "week",
            "team_key",
            "player_key",
            "selected_position",
            "is_starting",
            pl.col("points").alias("roster_points"),
        ])
        .join(
            stats_df.select([
                "league_key",
                "season",
                "week",
                "player_key",
                "fantasy_points",
            ]),
            on=["league_key", "season", "week", "player_key"],
            how="left",
        )
        .with_columns(
            pl.coalesce([pl.col("roster_points"), pl.col("fantasy_points")]).alias("player_points")
        )
    )

    non_null_points = player_weekly_points.select(pl.col("player_points").is_not_null().sum()).item()
    non_zero_points = player_weekly_points.select((pl.col("player_points") != 0).sum()).item()

    if non_null_points > 0 and non_zero_points > 0:
        position_weekly_points = (
            player_weekly_points
            .filter(pl.col("is_starting"))
            .group_by(["league_key", "season", "week", "selected_position"])
            .agg([
                pl.sum("player_points").alias("total_points"),
                pl.mean("player_points").alias("avg_points_per_player"),
                pl.n_unique("player_key").alias("players_counted"),
            ])
            .sort(["season", "week", "total_points"], descending=[False, False, True])
        )

        team_position_weekly_points = (
            player_weekly_points
            .filter(pl.col("is_starting"))
            .group_by(["league_key", "season", "week", "team_key", "selected_position"])
            .agg([
                pl.sum("player_points").alias("total_points"),
                pl.n_unique("player_key").alias("players_counted"),
            ])
            .sort(["season", "week", "team_key", "total_points"], descending=[False, False, False, True])
        )

        print("Weekly points by lineup position (all teams):")
        print(position_weekly_points)

        print("\nWeekly points by team and lineup position:")
        print(team_position_weekly_points)
    else:
        print("Player-level points are missing/zero in roster and stats tables.")
        print("Showing lineup position usage counts instead:")
        position_usage = (
            roster_df
            .filter(pl.col("is_starting"))
            .group_by(["league_key", "season", "week", "selected_position"])
            .agg(pl.n_unique("player_key").alias("players_counted"))
            .sort(["season", "week", "selected_position"])
        )
        print(position_usage)
else:
    print("Required tables are not available for player-level position analysis.")

Player-level points are missing/zero in roster and stats tables.
Showing lineup position usage counts instead:
shape: (102, 5)
┌──────────────┬────────┬──────┬───────────────────┬─────────────────┐
│ league_key   ┆ season ┆ week ┆ selected_position ┆ players_counted │
│ ---          ┆ ---    ┆ ---  ┆ ---               ┆ ---             │
│ str          ┆ i64    ┆ i64  ┆ str               ┆ u32             │
╞══════════════╪════════╪══════╪═══════════════════╪═════════════════╡
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ DEF               ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ QB                ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ RB                ┆ 24              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ TE                ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ W/R/T             ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ WR                ┆ 36              │
│ 461.l.717896 ┆ 2025   ┆ 2    ┆ DEF               ┆ 12              │
│ 461.l.717896 ┆ 2025

In [51]:
# Example 4 result viewer (run this cell after Example 4)
# Shows points-by-position when available, otherwise shows position usage.

if position_weekly_points is not None:
    print("Example 4: Weekly points by lineup position (top 50 rows)")
    print(position_weekly_points.head(50))

    if team_position_weekly_points is not None:
        print("\nExample 4: Weekly points by team and lineup position (top 50 rows)")
        print(team_position_weekly_points.head(50))
elif "position_usage" in globals() and position_usage is not None:
    print("Example 4: Position usage counts (player points missing in source data)")
    print(position_usage.head(50))
else:
    print("Example 4 has no computed output yet.")
    print("Run the 'Player-Level Aggregation by Position' cell first.")

Example 4: Position usage counts (player points missing in source data)
shape: (50, 5)
┌──────────────┬────────┬──────┬───────────────────┬─────────────────┐
│ league_key   ┆ season ┆ week ┆ selected_position ┆ players_counted │
│ ---          ┆ ---    ┆ ---  ┆ ---               ┆ ---             │
│ str          ┆ i64    ┆ i64  ┆ str               ┆ u32             │
╞══════════════╪════════╪══════╪═══════════════════╪═════════════════╡
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ DEF               ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ QB                ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ RB                ┆ 24              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ TE                ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ W/R/T             ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 1    ┆ WR                ┆ 36              │
│ 461.l.717896 ┆ 2025   ┆ 2    ┆ DEF               ┆ 12              │
│ 461.l.717896 ┆ 2025   ┆ 2    ┆ QB                ┆ 12      